1. **PDF Text Extractor:** Using PyMuPDF, extract all text from the first 20 pages of Socfinaf S.A annual report PDF; clean whitespace; count word frequency; save top 100 keywords to CSV. (Link: https://socfin.com/wp-content/uploads/2025/05/2024-Socfinaf-Annual-report.pdf )

In [12]:
!pip install pymupdf

In [13]:
import fitz  # PyMuPDF
import pandas as pd
import re
from collections import Counter

In [14]:
#Opening file locally
doc = fitz.open("2024-Socfinaf-Annual-report.pdf")

# Extracting text from first 20 pages
text = ""
for i in range(21):
    page = doc[i]
    text += page.get_text()

doc.close()

In [15]:
# Cleaning the text:
# converting to lowercase
text = text.lower()

# removing punctuation and numbers
text = re.sub(r'[^a-z\s]', ' ', text)

# removing extra spaces
text = re.sub(r'\s+', ' ', text).strip()


In [16]:
#Counting word frequency
words = text.split()
word_counts = Counter(words)

In [17]:
#Getting top 100 words
top_100 = word_counts.most_common(100)
#Saving top 100 words to a dataframe
df = pd.DataFrame(top_100, columns=["word", "frequency"])

#Saving words to CSV
df.to_csv("Top 100 keywords.csv", index=False)


**2. OOP for AI:** Build a DataLoader class that accepts a file path, auto-detects format (CSV/JSON/TXT), loads it, and returns a Pandas DataFrame or list of strings. Includes error handling and a _repr_ method.

In [18]:
import pandas as pd
import json
import os

In [19]:
class DataLoader:
    def __init__(self, file_path):
        self.file_path = file_path
        self.file_type = self._detect_file_type()
        self.data = None

    #Detecting file type based on file extension
    def _detect_file_type(self):
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"File not found: {self.file_path}") #Raise error if file path is invalid

        _, ext = os.path.splitext(self.file_path) #Split for example "data.csv" to ("data",".csv")

        ext = ext.lower() #Convert for example .CSV to .csv

        if ext == ".csv":
            return "csv"
        elif ext == ".json":
            return "json"
        elif ext == ".txt":
            return "txt"
        else:
            raise ValueError(f"Unsupported file type: {ext}")

    #Loading file based on detected type
    def load(self):
        try:
            if self.file_type == "csv":
                self.data = pd.read_csv(self.file_path)

            elif self.file_type == "json":
                # Try loading as DataFrame first
                try:
                    self.data = pd.read_json(self.file_path)
                except ValueError:
                # if JSON file is not tabular:
                    with open(self.file_path, "r", encoding="utf-8") as f:
                        loaded_data = json.load(f)
                        self.data = pd.DataFrame(loaded_data)

            elif self.file_type == "txt":
                with open(self.file_path, "r", encoding="utf-8") as f:
                    self.data = [line.strip() for line in f] # creates a list of lines for example ["line 1", "line 2", "line 3"]

            return self.data

        except Exception as e:
            raise RuntimeError(f"Error loading file: {e}") #if anything fails like corrupted file, we get a clear error message

    def __repr__(self):
        return f"DataLoader(file_path='{self.file_path}', file_type='{self.file_type}')"

In [20]:
Loader = DataLoader("Top 100 keywords.csv")
Data= Loader.load()
print(Loader)
Data.head()

DataLoader(file_path='Top 100 keywords.csv', file_type='csv')


,word,frequency
0,of,131
1,the,114
2,and,88
3,in,72
4,eur,56
